# `@staticmethod` 和 `@classmethod`

Python 类中常见的三类方法：

- 实例方法：第一个参数通常是 `self`，用于访问实例属性和实例方法。
- 类方法：使用 `@classmethod`，第一个参数通常是 `cls`，用于访问类属性或创建对象。
- 静态方法：使用 `@staticmethod`，不自动接收 `self` 或 `cls`，更像是放在类命名空间中的普通函数。

本 notebook 通过可运行的例子理解它们的区别。

## 一、先回顾实例方法

实例方法绑定到具体对象。调用 `user.introduce()` 时，Python 会自动把 `user` 作为第一个参数传入。

In [ ]:
class User:
    def __init__(self, name):
        self.name = name

    def introduce(self):
        return f"你好，我是 {self.name}"


user = User("小明")
user.introduce()

## 二、`@staticmethod`：不需要实例和类

静态方法不会自动接收 `self` 或 `cls`。它不能直接访问实例属性，也不能直接访问类属性；如果需要数据，就通过普通参数显式传入。

适合场景：逻辑与类的主题相关，但不依赖某个对象或某个类本身，例如格式校验、单位换算、纯计算等。

In [6]:
class User:
    @staticmethod
    def is_valid_name(name):
        return isinstance(name, str) and bool(name.strip())


print(User.is_valid_name("小明"))
print(User.is_valid_name("   "))

user = User()
print(user.is_valid_name("小红"))

True
False
True


静态方法也可以通过实例调用，但这并不意味着它获得了实例。下面的方法签名中没有 `self`，调用时参数完全由我们自己提供。

In [5]:
class Temperature:
    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return celsius * 9 / 5 + 32


Temperature.celsius_to_fahrenheit(0), Temperature().celsius_to_fahrenheit(0)

(32.0, 32.0)

## 三、`@classmethod`：绑定到类

类方法的第一个参数通常写作 `cls`，Python 会自动传入当前类。它可以访问类属性，也可以调用类的构造方法创建实例。

最常见用途是提供替代构造器，例如从字典、字符串或其他数据格式创建对象。

### 为什么类方法里的 `cls` 比直接写类名更好？

因为 `cls` 会随着继承发生变化。这样替代构造器在子类上调用时，会创建子类对象，而不是把类型写死为父类。

In [3]:
class User:
    role = "普通用户"

    def __init__(self, name):
        self.name = name

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"])

    @classmethod
    def describe_role(cls):
        return f"当前类的默认角色是：{cls.role}"


user = User.from_dict({"name": "小李"})
print(user.name)
print(User.describe_role())

class Admin(User):
    role = "管理员"


admin = Admin.from_dict({"name": "小王"})
print(type(admin).__name__)
print(admin.role)
print(Admin.describe_role())

小李
当前类的默认角色是：普通用户
Admin
管理员
当前类的默认角色是：管理员


## 四、三类方法对比

| 方法类型 | 装饰器 | 自动接收的第一个参数 | 能否直接访问实例属性 | 能否直接访问类属性 | 常见用途 |
| --- | --- | --- | --- | --- | --- |
| 实例方法 | 无 | `self` | 可以 | 可以 | 操作具体对象 |
| 类方法 | `@classmethod` | `cls` | 不可以 | 可以 | 替代构造器、操作类状态 |
| 静态方法 | `@staticmethod` | 无 | 不可以 | 不可以 | 与类主题相关的独立工具函数 |

一个简单的选择顺序：

1. 需要具体对象的数据吗？需要就用实例方法。
2. 需要当前类的信息或需要根据当前类创建对象吗？需要就用类方法。
3. 两者都不需要，只是逻辑上属于这个类吗？可以用静态方法。

## 五、放在一起看

下面的类同时展示三种方法：实例方法读取 `self.name`，类方法读取 `cls.species`，静态方法只处理传入的参数。

In [ ]:
class Animal:
    species = "动物"

    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name}发出声音"

    @classmethod
    def category(cls):
        return f"类别：{cls.species}"

    @staticmethod
    def is_name_valid(name):
        return isinstance(name, str) and len(name) > 0


cat = Animal("小猫")
print(cat.speak())
print(Animal.category())
print(Animal.is_name_valid("小猫"))

## 六、小练习

请实现一个 `Rectangle` 类：

- 实例方法 `area(self)`：返回当前矩形的面积。
- 类方法 `square(cls, side)`：根据边长创建一个正方形对象。
- 静态方法 `is_valid_side(side)`：判断边长是否为正数。

提示：类方法中使用 `cls(width, height)` 创建对象，这样继承 `Rectangle` 时也能保留多态性。

In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    @classmethod
    def square(cls, side):
        if not cls.is_valid_side(side):
            raise ValueError("边长必须是正数")
        return cls(side, side)

    @staticmethod
    def is_valid_side(side):
        return isinstance(side, (int, float)) and side > 0


square = Rectangle.square(5)
print(square.width, square.height, square.area())
print(Rectangle.is_valid_side(-1))

## 总结

- `@staticmethod`：不自动绑定实例或类，适合放在类中的独立工具函数。
- `@classmethod`：自动绑定当前类，适合访问类状态和实现替代构造器。
- `self` 代表对象，`cls` 代表类。
- 如果方法需要随着子类变化，优先考虑使用 `cls`，不要把父类名称写死。